[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/mobillity-univ/module-6-telling-the-story/notebook-6.3-shareable-presentations-powerpoint-and-web.ipynb)


# Shareable presentations: one FGC analysis, two formats

**The question.** We have a verified analysis of FGC evening-train service — one row per line for the 20:00–24:00 window on a representative weekday — and an executive brief that frames it as a Minto pyramid (main point, then reasons, then evidence). It has to reach the transport authority as a presentation. How do we build the deck straight from the notebook, instead of retyping every number into slides by hand?

**Two paths.** `python-pptx` writes a real PowerPoint file you can download and forward; `reveal.js` writes an interactive web deck that lives at a link and lets the audience hover over the chart. Same analysis, same pyramid — the output format is what changes.

**Where the data comes from.** The finding is the FGC evening-service analysis we have been carrying through this module, related to the per-line evening frequency from the Catalonia FGC GTFS timetable.

In [1]:
# --- Setup: install the library this notebook uses -------------------------
# On Google Colab the first run installs python-pptx; run locally it is a no-op
# when the package is already present. Safe to re-run.
import importlib.util, subprocess, sys

if importlib.util.find_spec("pptx") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=True)
print("SETUP_OK: python-pptx available")

SETUP_OK: python-pptx available


## 1. The verified analysis, carried in from previous lectures

This is the table already sitting in the notebook from previous lectures:

In [2]:
"""
The verified per-line evening-service table - carried forward, not recomputed.

Columns: line code, mode, evening departures, departures/hour, worst-direction wait (min).
"""
import pandas as pd

# One row per FGC line, exactly as produced and validated in story-6.1
# (sorted shortest evening wait first). This is the table already in the notebook.
line_table = pd.DataFrame(
    [
        ("FV", "funicular", 80, 20.0, 6),
        ("L7", "metro", 56, 14.0, 9),
        ("L12", "metro", 45, 11.2, 11),
        ("L6", "metro", 39, 9.8, 13),
        ("S1", "rail", 25, 6.2, 24),
        ("S2", "rail", 25, 6.2, 24),
        ("L8", "metro", 16, 4.0, 34),
        ("R63", "rail", 3, 0.8, 80),
        ("S8", "rail", 14, 3.5, 80),
        ("R53", "rail", 2, 0.5, 120),
        ("S3", "rail", 5, 1.2, 120),
        ("S4", "rail", 4, 1.0, 120),
        ("R5", "rail", 7, 1.8, 240),
        ("R6", "rail", 6, 1.5, 240),
        ("RL1", "rail", 3, 0.8, 240),
    ],
    columns=["route_short_name", "mode", "evening_departures", "dep_per_hour", "evening_wait_min"],
)

line_table

,route_short_name,mode,evening_departures,dep_per_hour,evening_wait_min
0,FV,funicular,80,20.0,6
1,L7,metro,56,14.0,9
2,L12,metro,45,11.2,11
3,L6,metro,39,9.8,13
4,S1,rail,25,6.2,24
5,S2,rail,25,6.2,24
6,L8,metro,16,4.0,34
7,R63,rail,3,0.8,80
8,S8,rail,14,3.5,80
9,R53,rail,2,0.5,120


Fifteen lines, 330 evening departures. The spread is stark: the funicular `FV` comes every 6 minutes, while `R5`, `R6` and `RL1` leave riders waiting up to 240 minutes — four hours — between trains. That unevenness is the story the deck has to carry, and it is now sitting in a table every slide can point at.

## 2. A shareable PowerPoint with `python-pptx`

The transport authority wants a file to open and pass around, so the first format is a PowerPoint. This is exactly where it is tempting to open PowerPoint and assemble the slides by hand — the move to skip. The brief and the numbers are already here, so we direct the assistant to write the code that turns the brief's pyramid into slides. We name the look we want — corporate blue titles, 24-point body text — so the deck comes out polished, not rough. A prompt for that:

Prompt to the assistant:

```markdown
Using python-pptx, build a PowerPoint deck from this FGC evening-service analysis and the executive brief. Follow the brief's pyramid: a title slide, a main-point slide with the headline finding, one slide per reason with its supporting chart or figure, and a closing recommendations slide. Give it a clean corporate look - blue #2563EB titles and 24-point body text - and save the .pptx file.
```

In [3]:
"""
Build a shareable PowerPoint deck from the FGC brief with python-pptx.
Data: brief (the pyramid, one slide each) + line_table (for the evidence chart).
"""
# 0. Settings you can change
DECK = "fgc-evening-service.pptx"
CHART = "fgc-evening-frequency.png"
ACCENT = "2563EB"          # corporate blue for the titles
BODY_PT = 24               # readable body text

# 1. The executive brief as a pyramid - one slide each
brief = [
    ("FGC Evening Service", "Weekday evening, 20:00-24:00 - prepared for the transport authority."),
    ("Evening service is uneven", "Riders on the thinner-served lines wait far longer for a train than riders on the core network."),
    ("The gap is large", "The best line runs every 6 minutes; the worst waits 240 minutes between trains - 40 times longer."),
    ("Six lines are barely served", "R53, S3, S4, R5, R6 and RL1 wait two hours or more between evening trains."),
    ("The imbalance is structural", "The best line runs 20 trains an hour; the worst runs one every two hours."),
    ("Recommendation", "Add evening trips on the lines waiting two hours or more, then re-check after the next timetable change."),
]

# 2. Render the evidence chart the "gap" slide will carry
import matplotlib
import matplotlib.pyplot as plt
ranked = line_table.sort_values("evening_wait_min")
colors = ["#10B981" if w <= 15 else "#F59E0B" if w < 120 else "#DC2626" for w in ranked.evening_wait_min]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(ranked.route_short_name, ranked.evening_wait_min, color=colors)
ax.invert_yaxis()
ax.set_xlabel("Minutes between trains - worst direction, evening 20:00-24:00")
ax.set_title("FGC evening service is uneven by line")
for i, w in enumerate(ranked.evening_wait_min):
    ax.text(w + 3, i, f"{int(w)}m", va="center", fontsize=9)
plt.tight_layout(); plt.savefig(CHART, dpi=120); plt.close(fig)

# 3. Build the slides: bold blue title, 24pt body, chart on the "gap" slide
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
prs = Presentation()
blank = prs.slide_layouts[6]
for heading, line in brief:
    slide = prs.slides.add_slide(blank)
    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(9), Inches(1)).text_frame
    title.text = heading
    trun = title.paragraphs[0].runs[0]
    trun.font.size = Pt(32); trun.font.bold = True; trun.font.color.rgb = RGBColor.from_string(ACCENT)
    body = slide.shapes.add_textbox(Inches(0.5), Inches(1.6), Inches(9), Inches(1.3)).text_frame
    body.word_wrap = True; body.text = line
    body.paragraphs[0].runs[0].font.size = Pt(BODY_PT)
    if heading == "The gap is large":
        slide.shapes.add_picture(CHART, Inches(0.9), Inches(3.1), width=Inches(8))
prs.save(DECK)

print("CHART_SAVED:", CHART)
print("DECK_SAVED:", DECK)
print("SLIDE_COUNT:", len(prs.slides._sldIdLst))

CHART_SAVED: fgc-evening-frequency.png
DECK_SAVED: fgc-evening-service.pptx
SLIDE_COUNT: 6


It runs and saves a real `.pptx`: six slides, the evidence chart embedded on the "gap" slide, every figure carried straight from the verified table. Two misconceptions fall away at once — slide creation did not have to be manual, and, because we specified the template, colours and font sizes, the generated deck looks polished.

## 3. An interactive web deck with `reveal.js`

Same analysis, same pyramid — but when the audience benefits from a link and from exploring the numbers themselves, a web deck fits better. We ask for a `reveal.js` presentation and, this time, embed the evening-frequency chart as an interactive Plotly figure rather than a static image. A prompt for that:

Prompt to the assistant:

```markdown
Now build the same presentation as an interactive web deck with reveal.js. Keep the pyramid - the headline, the three reasons, and the recommendation - and embed the evening-frequency chart as an interactive Plotly figure the audience can hover over. Use a web layout with a dark #111827 header on a white body, and save the HTML file.
```

In [4]:
"""
Build the same presentation as an interactive web deck with reveal.js.
Data: brief (the pyramid) + line_table (for the interactive Plotly chart).
"""
# 0. Settings you can change
DECK = "fgc-evening-service.html"
HEADER = "#111827"   # dark web header
BODY = "#FFFFFF"     # white body

# 1. The same pyramid - one <section> per slide
brief = [
    ("FGC Evening Service", "Weekday evening, 20:00-24:00 - prepared for the transport authority."),
    ("Evening service is uneven", "Riders on the thinner-served lines wait far longer for a train than the core network."),
    ("The gap is large", "The best line runs every 6 minutes; the worst waits 240 minutes - 40 times longer."),
    ("Six lines are barely served", "R53, S3, S4, R5, R6 and RL1 wait two hours or more between evening trains."),
    ("The imbalance is structural", "The best line runs 20 trains an hour; the worst runs one every two hours."),
    ("Recommendation", "Add evening trips on the lines waiting two hours or more, then re-check after the next timetable change."),
]

# 2. The evidence chart as an INTERACTIVE Plotly figure (hover shows the exact wait)
import plotly.graph_objects as go
ranked = line_table.sort_values("evening_wait_min")
colors = ["#10B981" if w <= 15 else "#F59E0B" if w < 120 else "#DC2626" for w in ranked.evening_wait_min]
fig = go.Figure(go.Bar(
    x=ranked.evening_wait_min, y=ranked.route_short_name, orientation="h",
    marker_color=colors,
    hovertemplate="Line %{y}: %{x} minutes between evening trains<extra></extra>"))
fig.update_layout(
    title="FGC evening service is uneven by line",
    xaxis_title="Minutes between trains - worst direction, evening 20:00-24:00",
    yaxis=dict(title="Line", autorange="reversed"), template="simple_white")

# 3. Assemble the reveal.js deck: dark header, white body, chart on the evidence slide
chart_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
def section(heading, line, extra=""):
    return f"<section><h2>{heading}</h2><p>{line}</p>{extra}</section>"
slides = "".join(section(h, l, chart_html if h == "The gap is large" else "") for h, l in brief)
html = f'''<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/reveal.js@4/dist/reveal.css">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/reveal.js@4/dist/theme/white.css">
<style>.reveal h2{{color:{HEADER}}} body{{background:{BODY}}}</style>
</head><body><div class="reveal"><div class="slides">{slides}</div></div>
<script src="https://cdn.jsdelivr.net/npm/reveal.js@4/dist/reveal.js"></script>
<script>Reveal.initialize();</script></body></html>'''
open(DECK, "w").write(html)

print("DECK_SAVED:", DECK)
print("SLIDE_COUNT:", len(brief))
fig

DECK_SAVED: fgc-evening-service.html
SLIDE_COUNT: 6


The code writes the HTML deck and builds the chart it carries. Unlike the static image in the PowerPoint, this chart is interactive: the audience can hover over any line to read its exact evening wait. That interactivity is the reason to reach for `reveal.js`.

## Read the answer

We started with a finding that lived in code and a table — FGC's evening service is uneven, with the outer lines waiting up to four hours between trains — and needed it in front of the transport authority. From the same notebook we produced two decks without rebuilding anything by hand: a shareable `.pptx` with `python-pptx`, and an interactive web deck with `reveal.js`. The analysis never left its source, and every number on every slide came straight from the verified table. Which format you reach for is a question of audience and context — a file to forward, or a link to explore — not of redoing the work.